# Ideas to reduce memory usage

- Call WERK & MATNR data iteratively -> Estimation: reduces the load by A LOT
    - **TODO**: change code that it first get's all the combinations from the EMS and the iterate over the data
- Change dtypes of pd.DataFrames -> Estimation: will for sure help but not as much as going throgh the data iteratively
    - **TODO**: check with business, what values for what column are to be expected. E.g. if a float 32 / 16 is enough for the required amount of material
- Drop unused columns -> Estimation: might reduce the data quite a bit
    - **TODO**: check with business, what data is required in the output table
- Last resort: open a ticket and request more RAM

# Ideas to optimize speed
- Parallelize to more CPU cores
    - **TODO**: adapt code to use multiprocessing and reduce memory usage (see above) / increase MLWB memory, such that we do not run out of memory
- Maybe algorithmically there are some improvements that can be done
    - **TODO**: invest (maybe a lot) time to think about the algorithm used and if it can be improved

# Necessary imports

In [ ]:
# Connect to Celonis and Import Libraries used
import gc
from pycelonis import get_celonis
from pycelonis import pql
from pycelonis.ems import ColumnTransport, ColumnType
import pandas as pd
from datetime import datetime
from typing import List, Tuple
from collections import namedtuple
from loguru import logger
import os
from tqdm.notebook import tqdm

celonis = get_celonis(permissions=False)

# Configuration

In [64]:
WerkAndMatnr = namedtuple('WerkAndMatnr', ["werk","matnr"])
list_of_werk_and_matnrs_for_testing = [
    WerkAndMatnr(werk='2260', matnr='FRS132668'),
]

In [65]:
# Identify if it is a scheduled run. This is e.g. important to not use tqdm. tqdm breaks the automatic execution.
SCHEDULED_RUN = bool(os.environ.get("CELONIS_NOTEBOOK_EXECUTION")=='true')

# Constants

In [66]:
FORECAST_VALUE_COLUMN = "PBED.PLNMG"
LEFTOVER_QTY_COLUMN = "Leftover QTY"
TEST_WERK = ",".join([f"'{e.werk}'" for e in list_of_werk_and_matnrs_for_testing])
TEST_MATNR = ",".join([f"'{e.matnr}'" for e in list_of_werk_and_matnrs_for_testing])

# Feature flags
#### FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX
For validation, this flag filters down the amount of data to only WERKS and MATNR specified in list_of_werk_and_matnrs_for_testing. Otherwise, the produced data would go into the GB region and the notebook won't be able to handle it with respect to RAM

#### SAVE_TO_PICKLE_FILES
To test locally, this flag saves the usage & forecast data to .pkl files to download & use them in local development environments

In [67]:
FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX = False
SAVE_TO_PICKLE_FILES = False

# Load data model from Celonis

In [68]:
# Get IM Data Model using DM Key in URL
datapool = celonis.data_integration.get_data_pools().find('DATA POOL NAME HERE')   # Add data pool name
datamodel = datapool.get_data_models().find('DATA MODEL NAME HERE')   # Add data model name

# Create required data

## Create MARC/VBBE1 table where OMENG > O then use VBBE.OMENG

In [ ]:
# Create MARC/VBBE1 Table for Sales Orders
logger.info("Start loading MARC/VBBE1 table")

q1 = pql.PQL()
q1 += pql.PQLColumn(name = "MATNR", query = "VBBE.MATNR")
q1 += pql.PQLColumn(name = "WERKS", query = "VBBE.WERKS")
q1 += pql.PQLColumn(name = "DATE", query = "VBBE.MBDAT")
q1 += pql.PQLColumn(name = "SO", query = "VBBE.VBELN || '/'  || VBBE.POSNR || '/' || VBBE.ETENR")
q1 += pql.PQLColumn(name = "QTY_NEEDED", query = "VBBE.OMENG")
q1 += pql.PQLColumn(name = "MARC.VRMOD", query = "MARC.VRMOD")
q1 += pql.PQLColumn(name = "MARC.VINT2", query ="TO_INT(MARC.VINT2)")
q1 += pql.PQLColumn(name = "MARC.VINT1", query ="TO_INT(MARC.VINT1)")
q1 += pql.PQLColumn(name = "DATE_START", query ="CASE WHEN MARC.VRMOD IN ('1', '2', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , VBBE.MBDAT , -1*(TO_INT(MARC.VINT1)) ) ELSE VBBE.MBDAT END")
q1 += pql.PQLColumn(name = "DATE_END", query ="CASE WHEN MARC.VRMOD IN ('2', '3', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , VBBE.MBDAT , TO_INT(MARC.VINT2) ) ELSE VBBE.MBDAT END")
q1 += pql.PQLFilter(query="FILTER VBBE.OMENG > 0")
q1 += pql.PQLFilter(query="FILTER VBAP.ABGRU IS NULL")
# Even completed orders can have a qty from LIPS if no Goods issue have been posted
#q += pql.PQLFilter(query="FILTER VBAP.SALES_ORDER_ITEM_STATUS != 'C'")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q1 += pql.PQLFilter(query=f"FILTER VBBE.MATNR IN ({TEST_MATNR});") 
    q1 += pql.PQLFilter(query=f"FILTER VBBE.WERKS IN ({TEST_WERK});")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data
q1 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

# # Get MARC/VBBE Table
vbbe_table1 = datamodel.export_data_frame(query=q1)
vbbe_table1['MRP_ELEMENT'] = 'Sales Order'

logger.info("End loading MARC/VBBE1 table")

## Create MARC/VBBE2 where OMENG = O then use LIPS.LGMNG

In [ ]:
# Create MARC/VBBE2 Table for Sales Orders
logger.info("Start loading MARC/VBBE2 table")

q2 = pql.PQL()
q2 += pql.PQLColumn(name = "MATNR", query = "VBBE.MATNR")
q2 += pql.PQLColumn(name = "WERKS", query = "VBBE.WERKS")
q2 += pql.PQLColumn(name = "DATE", query = "VBBE.MBDAT")
q2 += pql.PQLColumn(name = "SO", query = "VBBE.VBELN || '/'  || VBBE.POSNR || '/' || VBBE.ETENR")
q2 += pql.PQLColumn(name = "QTY_NEEDED", query = "PU_SUM(VBAP, LIPS.LGMNG)")
q2 += pql.PQLColumn(name = "MARC.VRMOD", query = "MARC.VRMOD")
q2 += pql.PQLColumn(name = "MARC.VINT2", query ="TO_INT(MARC.VINT2)")
q2 += pql.PQLColumn(name = "MARC.VINT1", query ="TO_INT(MARC.VINT1)")
q2 += pql.PQLColumn(name = "DATE_START", query ="CASE WHEN MARC.VRMOD IN ('1', '2', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , VBBE.MBDAT , -1*(TO_INT(MARC.VINT1)) ) ELSE VBBE.MBDAT END")
q2 += pql.PQLColumn(name = "DATE_END", query ="CASE WHEN MARC.VRMOD IN ('2', '3', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , VBBE.MBDAT , TO_INT(MARC.VINT2) ) ELSE VBBE.MBDAT END")
q2 += pql.PQLFilter(query="FILTER VBBE.OMENG = 0")
q2 += pql.PQLFilter(query="FILTER PU_SUM(VBAP, LIPS.LGMNG) IS NOT NULL")
q2 += pql.PQLFilter(query="FILTER VBAP.ABGRU IS NULL")
# Even completed orders can have a qty from LIPS if no Goods issue have been posted
#q += pql.PQLFilter(query="FILTER VBAP.SALES_ORDER_ITEM_STATUS != 'C'")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q2 += pql.PQLFilter(query=f"FILTER VBBE.MATNR IN ({TEST_MATNR});") 
    q2 += pql.PQLFilter(query=f"FILTER VBBE.WERKS IN ({TEST_WERK});")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data
q2 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

# # Get MARC/VBBE Table
vbbe_table2 = datamodel.export_data_frame(query=q2)
vbbe_table2['MRP_ELEMENT'] = 'Sales Order'

logger.info("End loading MARC/VBBE2 table")

In [ ]:
vbbe_table2.dtypes

## Create MARC/RESB table

In [ ]:
# Create MARC/RESB Table for Reservations
q3 = pql.PQL()
q3 += pql.PQLColumn(name = "MATNR", query = "RESB.MATNR")
q3 += pql.PQLColumn(name = "WERKS", query = "RESB.WERKS")
q3 += pql.PQLColumn(name = "DATE", query = "RESB.BDTER")
q3 += pql.PQLColumn(name = "Res", query = "RESB.RSNUM || '/'  || RESB.RSPOS")
q3 += pql.PQLColumn(name = "QTY_NEEDED", query = "RESB.BDMNG - RESB.ENMNG")
q3 += pql.PQLColumn(name = "MARC.VRMOD", query = "MARC.VRMOD")
q3 += pql.PQLColumn(name = "MARC.VINT2", query ="TO_INT(MARC.VINT2)")
q3 += pql.PQLColumn(name = "MARC.VINT1", query ="TO_INT(MARC.VINT1)")
q3 += pql.PQLColumn(name = "DATE_START", query ="CASE WHEN MARC.VRMOD IN ('1', '2', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , RESB.BDTER , -1*(TO_INT(MARC.VINT1)) ) ELSE RESB.BDTER END")
q3 += pql.PQLColumn(name = "DATE_END", query ="CASE WHEN MARC.VRMOD IN ('2', '3', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , RESB.BDTER , TO_INT(MARC.VINT2) ) ELSE RESB.BDTER END")
q3 += pql.PQLFilter(query="FILTER RESB.BDMNG - RESB.ENMNG > 0")
q3 += pql.PQLFilter(query="FILTER RESB.XLOEK IS NULL")
q3 += pql.PQLFilter(query="FILTER RESB.KZEAR IS NULL")
q3 += pql.PQLFilter(query="FILTER (RESB.POSTP IN ('L', 'R') OR RESB.POSTP IS NULL)")
q3 += pql.PQLFilter(query="FILTER RESB.SCHGT IS NULL")
q3 += pql.PQLFilter(query="FILTER RESB.DUMPS IS NULL")
q3 += pql.PQLFilter(query="FILTER RESB.BDMNG - RESB.ENMNG > 0")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q3 += pql.PQLFilter(query=f"FILTER RESB.MATNR IN ({TEST_MATNR});") 
    q3 += pql.PQLFilter(query=f"FILTER RESB.WERKS IN ({TEST_WERK});")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data
q3 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

# Get MARC/RESB Table
resb_table = datamodel.export_data_frame(query=q3)
resb_table['MRP_ELEMENT'] = 'Reservation'

In [ ]:
resb_table

## Create MARC/EKET table

In [ ]:
#Create MARC/EKET Table for Orders
q4 = pql.PQL()
q4 += pql.PQLColumn(name = "MATNR", query = "MARC.MATNR")
q4 += pql.PQLColumn(name = "WERKS", query = "MARC.WERKS")
q4 += pql.PQLColumn(name = "DATE", query = "EKET.MBDAT")
q4 += pql.PQLColumn(name = "SO", query = "EKET.EBELN || '/' ||EKET.EBELP || '/' || EKET.ETENR")
q4 += pql.PQLColumn(name = "QTY_NEEDED", query = "(EKET.MNG02 - EKET.GLMNG)")
q4 += pql.PQLColumn(name = "MARC.VRMOD", query = "MARC.VRMOD")
q4 += pql.PQLColumn(name = "MARC.VINT2", query ="TO_INT(MARC.VINT2)")
q4 += pql.PQLColumn(name = "MARC.VINT1", query ="TO_INT(MARC.VINT1)")
q4 += pql.PQLColumn(name = "DATE_START", query = "CASE WHEN MARC.VRMOD IN ('1', '2', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , EKET.MBDAT , -1*(TO_INT(MARC.VINT1)) ) ELSE EKET.MBDAT END")
q4 += pql.PQLColumn(name = "DATE_END", query = "CASE WHEN MARC.VRMOD IN ('2', '3', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , EKET.MBDAT , TO_INT(MARC.VINT2) ) ELSE EKET.MBDAT END")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q4 += pql.PQLFilter(query=f"FILTER EKET.MATNR IN ({TEST_MATNR});") 
    q4 += pql.PQLFilter(query=f"FILTER EKET.WERKS IN ({TEST_WERK});")
q4 += pql.PQLFilter(query="FILTER (EKET.MNG02 - EKET.GLMNG) > 0")
q4 += pql.PQLFilter(query="FILTER LFA1.WERKS NOT IN (NULL);")
q4 += pql.PQLFilter(query="FILTER EKPO.LOEKZ IN (NULL);")
q4 += pql.PQLFilter(query="FILTER EKPO.EGLKZ IN (NULL);")
q4 += pql.PQLFilter(query="FILTER EKET.WEPOS = 'X';")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data
q4 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

#Get EKET Table
eket_table = datamodel.export_data_frame(query=q4)
eket_table['MRP_ELEMENT'] = 'Order'

## Create MARC/EBAN table

In [ ]:
#Create MARC/EBAN Table for Requisitions
q5 = pql.PQL()
q5 += pql.PQLColumn(name = "MATNR", query = "MARC.MATNR")
q5 += pql.PQLColumn(name = "WERKS", query = "MARC.WERKS")
q5 += pql.PQLColumn(name = "DATE", query = "EBAN.LFDAT")
q5 += pql.PQLColumn(name = "SO", query = "EBAN.BANFN || '/' || EBAN.BNFPO")
q5 += pql.PQLColumn(name = "QTY_NEEDED", query = "(EBAN.MENGE_SENT - EBAN.BSMNG_SENT)")
q5 += pql.PQLColumn(name = "MARC.VRMOD", query = "MARC.VRMOD")
q5 += pql.PQLColumn(name = "MARC.VINT2", query ="TO_INT(MARC.VINT2)")
q5 += pql.PQLColumn(name = "MARC.VINT1", query ="TO_INT(MARC.VINT1)")
q5 += pql.PQLColumn(name = "DATE_START", query = "CASE WHEN MARC.VRMOD IN ('1', '2', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , EBAN.LFDAT , -1*(TO_INT(MARC.VINT1)) ) ELSE EBAN.LFDAT END")
q5 += pql.PQLColumn(name = "DATE_END", query = "CASE WHEN MARC.VRMOD IN ('2', '3', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , EBAN.LFDAT , TO_INT(MARC.VINT2) ) ELSE EBAN.LFDAT END")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q5 += pql.PQLFilter(query=f"FILTER EBAN.MATNR IN ({TEST_MATNR});") 
    q5 += pql.PQLFilter(query=f"FILTER EBAN.WERKS IN ({TEST_WERK});")
q5+= pql.PQLFilter(query="FILTER (EBAN.MENGE_SENT - EBAN.BSMNG_SENT) > 0;")
q5 += pql.PQLFilter(query="FILTER EBAN.LOEKZ IN (NULL) OR EBAN.LOEKZ <> 'X';")
q5 += pql.PQLFilter(query="FILTER EBAN.EBAKZ IN (NULL) OR EBAN.EBAKZ <> 'X';")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data
q5 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

#Get EBAN Table
eban_table = datamodel.export_data_frame(query=q5)
eban_table['MRP_ELEMENT'] = 'Requisition'

## Create MARC/PLAF table

In [ ]:
#Create MARC/PLAF Table for Stock Transfers
q6 = pql.PQL()
q6 += pql.PQLColumn(name = "MATNR", query = "MARC.MATNR")
q6 += pql.PQLColumn(name = "WERKS", query = "MARC.WERKS")
q6 += pql.PQLColumn(name = "DATE", query = "PLAF.PSTTR")
q6 += pql.PQLColumn(name = "SO", query = "PLAF.PLNUM")
q6 += pql.PQLColumn(name = "QTY_NEEDED", query = "PLAF.GSMNG")
q6 += pql.PQLColumn(name = "MARC.VRMOD", query = "MARC.VRMOD")
q6 += pql.PQLColumn(name = "MARC.VINT2", query ="TO_INT(MARC.VINT2)")
q6 += pql.PQLColumn(name = "MARC.VINT1", query ="TO_INT(MARC.VINT1)")
q6 += pql.PQLColumn(name = "DATE_START", query = "CASE WHEN MARC.VRMOD IN ('1', '2', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , PLAF.PSTTR , -1*(TO_INT(MARC.VINT1)) ) ELSE PLAF.PSTTR END")
q6 += pql.PQLColumn(name = "DATE_END", query = "CASE WHEN MARC.VRMOD IN ('2', '3', '4') THEN ADD_WORKDAYS ( WORKDAY_CALENDAR ( TFACS , '01' ) , PLAF.PSTTR , TO_INT(MARC.VINT2) ) ELSE PLAF.PSTTR END")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q6 += pql.PQLFilter(query=f"FILTER PLAF.MATNR IN ({TEST_MATNR});") 
    q6 += pql.PQLFilter(query=f"FILTER PLAF.PWWRK IN ({TEST_WERK});")
q6+= pql.PQLFilter(query="FILTER PLAF.GSMNG > 0")
q6 += pql.PQLFilter(query="FILTER PLAF.PLWRK <> PLAF.PWWRK;")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data
q6 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

#Get PLAF Table
plaf_table = datamodel.export_data_frame(query=q6)
plaf_table['MRP_ELEMENT'] = 'Stock Transfer'

## Merge tables

In [76]:
# Concat VBBE, EKET, EBAN, PLAF Tables
usage_tables = [vbbe_table1, vbbe_table2, resb_table, eban_table, eket_table, plaf_table]
usage_df = pd.concat(usage_tables).sort_values(by=["MATNR", "WERKS","DATE"])

In [77]:
# Free up resources
del vbbe_table1, vbbe_table2, resb_table, eban_table, eket_table, plaf_table
gc.collect()

3592

In [78]:
categorical_columns = ["MATNR","WERKS","MARC.VRMOD","MARC.VINT2","MARC.VINT1","MRP_ELEMENT"]
usage_df[categorical_columns] = usage_df[categorical_columns].astype("category")

In [79]:
if SAVE_TO_PICKLE_FILES: usage_df.to_pickle("usage.pkl")

## Create PBED table (Independent Requirements / Forecast)

In [ ]:
# Create PBED Table for Independent Requirements
q7 = pql.PQL()
q7 += pql.PQLColumn(name = "PBED.MANDT", query = "PBED.MANDT")
q7 += pql.PQLColumn(name = "MATNR", query = "PBED.MATNR")
q7 += pql.PQLColumn(name = "WERKS", query = "PBED.WERKS")
q7 += pql.PQLColumn(name = "PBED.PBDNR", query = "PBED.PBDNR")
q7 += pql.PQLColumn(name = "PBED.BDZEI", query = "PBED.BDZEI")
q7 += pql.PQLColumn(name = "PBED.PDATU", query = "PBED.PDATU")
q7 += pql.PQLColumn(name = "PBED.PLNMG", query = "PBED.PLNMG")
q7 += pql.PQLColumn(name = "PBED.ENTMG", query = "PBED.ENTMG")
q7 += pql.PQLColumn(name = "PBED.MEINS", query = "PBED.MEINS")
# Real available capacity is PBED.PLNMG - PBED.ENTMG
q7 += pql.PQLColumn(name = "PBED.PLNMGminusPBED.ENTMG", query = "PBED.PLNMG - PBED.ENTMG")
q7 += pql.PQLColumn(name = "PBED.PERXX", query = "PBED.PERXX")
q7 += pql.PQLFilter(query="FILTER PBED.PLNMG > 0;")
q7 += pql.PQLFilter(query="FILTER PBED.ENTLI IN ('1','2','3');")
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    q7 += pql.PQLFilter(query=f"FILTER PBED.MATNR IN ({TEST_MATNR});")
    q7 += pql.PQLFilter(query=f"FILTER PBED.WERKS IN ({TEST_WERK});")
# VRMOD can be empty -> to reduce data load, we drop these cases when querying the data -> commented out because we also need not depleted Ind. Req.
# q2 += pql.PQLFilter(query="FILTER MARC.VRMOD IS NOT NULL;")

# Get PBED Table
# Independent Requirements -> Forecast
forecast_df = datamodel.export_data_frame(query=q7).sort_values(by=["MATNR", "WERKS","PBED.BDZEI","PBED.PDATU"])

In [ ]:
forecast_df

In [82]:
forecast_df[LEFTOVER_QTY_COLUMN] = forecast_df[FORECAST_VALUE_COLUMN]

In [83]:
if SAVE_TO_PICKLE_FILES: forecast_df.to_pickle("forecast.pkl")

# Calculate forecasts

In [84]:
def split_forecast(forecast_df: pd.DataFrame, date: datetime):
    """Split forecast into past and future forecast

    Args:
        forecast_df (pd.DataFrame): DataFrame with forecast
        date (datetime): Date to split forecast into past (<=) and future (>)

    Returns:
        tuple(pd.DataFrame, pd.DataFrame): Past and future forecast DataFrames
    """

    past_forecast_df = forecast_df[
        (forecast_df[LEFTOVER_QTY_COLUMN] > 0) &
        (forecast_df["PBED.PDATU"] <= date)
    ]

    future_forecast_df = forecast_df[
        (forecast_df[LEFTOVER_QTY_COLUMN] > 0) &
        (forecast_df["PBED.PDATU"] > date)
    ]

    return past_forecast_df, future_forecast_df

In [85]:
def filter_to_relevant_forecast(forecast_df:pd.DataFrame, start:datetime, end:datetime) -> pd.DataFrame:
    """Filter forecast to relevant dates given by usage START_DATE and END_DATE

    Args:
        forecast_df (pd.DataFrame): Forecast DataFrame
        start (datetime): Start of relevant forecast values
        end (datetime): End of relevant forecast values

    Returns:
        pd.DataFrame: Forecast DataFrame reduced to the relevant dates
    """
    filtered_forecast_df = forecast_df[
        (forecast_df["PBED.PDATU"] >= start) &
        (forecast_df["PBED.PDATU"] <= end)
    ]

    return filtered_forecast_df

In [86]:
def order_forecasts_according_to_vrmod(past_forecast_df:pd.DataFrame, future_forecast_df:pd.DataFrame, vrmod:str) -> pd.DataFrame:
    """Sort forecast according to vrmod. See below for more information what which vrmod means.

    Args:
        past_forecast_df (pd.DataFrame): Forecast DataFrame for the past
        future_forecast_df (pd.DataFrame): Forecast DataFrame for the future
        vrmod (str): vrmod value from MARC.VRMOD 

    Raises:
        ValueError: If vrmod is not 1, 2, 3, 4, then this error is raised.

    Returns:
        pd.DataFrame: DataFrame with forecast sorted according to vrmod
    """

    if vrmod == "1":
        # Backward Consumption from start_date until mrp_date
        return past_forecast_df.iloc[::-1]

    if vrmod == "2":
        # past_forecast_df from start_date until mrp_date; future_forecast_df from mrp_date to end_date
        return pd.concat([past_forecast_df.iloc[::-1], future_forecast_df])

    if vrmod == "3":
        # Forward consumption from mrp_date to end_date
        return future_forecast_df

    if vrmod == "4":
        # Forward consumption from mrp_date to end_date; past_forecast_df from start_date until mrp_date
        return pd.concat([future_forecast_df, past_forecast_df.iloc[::-1]]) 

    raise ValueError(f"vrmod had a value of {vrmod} and that is not supported. Please check. Only 1, 2, 3, 4 are supported.")        
    

In [87]:
def consume_forecast(
        forecast_df:pd.DataFrame, 
        ordered_forecast_df: pd.DataFrame, 
        usage_row: pd.DataFrame, 
    ) -> List[pd.Series]:
    """Consume forecast according to the ordered forecast DataFrame. 
    This means that the forecast is reduced by the amount needed and 
    the assigned quantity to the usage is increased by this amount.

    Args:
        forecast_df (pd.DataFrame): The original DataFrame that holds the forecast. 
                                    This is used to communicate the change to the next loop 
                                    and persist the change in forecast.
        ordered_forecast_df (pd.DataFrame): The ordered forecast DataFrame that is used to go through the forecast according to vrmod.
        usage_row (pd.Series): The row of the usage DataFrame that we are currently trying to fulfill with forecasts.

    Returns:
        List(pd.Series): _description_
    """
    qty_needed, so = usage_row[["QTY_NEEDED", "SO"]]

    qty_assigned = 0
    consumed_forecasts = []

    for idx, available_forecast in ordered_forecast_df.iterrows():
        plnmg = available_forecast[LEFTOVER_QTY_COLUMN]
        # Case 1: forecast row is bigger than required quantity
        if plnmg >= qty_needed:
            qty_assigned += qty_needed

            forecast_df.loc[idx, LEFTOVER_QTY_COLUMN] = plnmg - qty_needed

            available_forecast["Assigned Qty"] = qty_needed 
            available_forecast["SO"] = so
            consumed_forecasts.append(available_forecast)

            return consumed_forecasts

        # Case 2: forecast row is smaller than required quantity
        if plnmg < qty_needed:
            qty_assigned += plnmg
            qty_needed -= plnmg
            forecast_df.loc[idx, LEFTOVER_QTY_COLUMN] = 0

            available_forecast["Assigned Qty"] = plnmg 
            available_forecast["SO"] = so
            consumed_forecasts.append(available_forecast)

    return consumed_forecasts

In [88]:
# Reduce the data -> only werk & matnr that have a respective forecast are considered
dict_of_forecast_werk_matnr_df = {key:df for key,df in forecast_df.groupby(["WERKS", "MATNR"])}
werk_matnr_with_forecasts = set(dict_of_forecast_werk_matnr_df.keys())
dict_of_usage_werk_matnr_df = {key:df for key,df in usage_df.groupby(["WERKS", "MATNR"]) if key in werk_matnr_with_forecasts}

**IMPORTANT**  
In the following codeblock we have a for loop.  
It is important to grap first the forecast AND THEN the usage.  
Reason: it might happen, that there is a forecast but no usage -> this would lead to the case, that the forecast is not added to the list via all_unassigned_forecasts_werk_matnr.append(forecast_werk_matnr_df), since the for loop is never entered.
Thus, this forecast would not be included in the result.

In [ ]:
# Sort properly
forecast_df = forecast_df.sort_values(by=["PBED.PDATU"])
usage_df = usage_df.sort_values(by=["DATE"])

# Create lists that will hold the final data
all_werk_matnr_usage_with_forecasts = []
all_unassigned_forecasts_werk_matnr = []

# Go through all the werk & matnr combinations
# disable = SCHEDULED_RUN -> important, since progress bar breaks the scheduled run

for (werk, matnr), forecast_werk_matnr_df in tqdm(dict_of_forecast_werk_matnr_df.items(), disable=SCHEDULED_RUN):
    grouped_usage_df = dict_of_usage_werk_matnr_df.get((werk, matnr), pd.DataFrame())
    
    consumed_forecasts = []
    # Go through all the usages for the current werk & matnr
    for _, usage_row in grouped_usage_df.iterrows():
        # Circuit breaker: if no forecast exists because it was all consumed in previous usage
        if not len(forecast_werk_matnr_df) > 0:
            break

        available_forecast = filter_to_relevant_forecast(forecast_werk_matnr_df, usage_row["DATE_START"], usage_row["DATE_END"])
        past_forecast_df, future_forecast_df = split_forecast(
            available_forecast, usage_row["DATE"])
        ordered_forecast_df = order_forecasts_according_to_vrmod(past_forecast_df, future_forecast_df, usage_row["MARC.VRMOD"])
        consumed_forecast = consume_forecast(forecast_werk_matnr_df, ordered_forecast_df, usage_row)
        consumed_forecasts.extend(consumed_forecast)

        # Drop rows that have no forecast quantity to assign left
        forecast_werk_matnr_df = forecast_werk_matnr_df[forecast_werk_matnr_df[LEFTOVER_QTY_COLUMN] > 0]

    all_unassigned_forecasts_werk_matnr.append(forecast_werk_matnr_df)

    if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
        consumed_forecasts_df = pd.DataFrame(consumed_forecasts)
        if len(consumed_forecasts) == 0: continue # If no forecasts were consumed, then skip this werk & matnr combination

        consumed_forecasts_df = consumed_forecasts_df[["SO","PBED.PDATU", "PBED.PBDNR", "Assigned Qty"]]
        consumed_forecasts_df["SO"] = consumed_forecasts_df["SO"].astype("string")
        usage_with_forecasts = usage_df.merge(consumed_forecasts_df, how="left", on="SO")
        all_werk_matnr_usage_with_forecasts.append(usage_with_forecasts)

all_unassigned_forecasts_werk_matnr_df = pd.concat(all_unassigned_forecasts_werk_matnr)
all_unassigned_forecasts_werk_matnr_df.sort_values(by=["WERKS","MATNR"], inplace=True)

In [90]:
if FILTER_TO_WERK_AND_MATNR_AND_SAVE_TO_XLSX:
    if all_werk_matnr_usage_with_forecasts:
        all_werk_matnr_usage_with_forecasts_df = pd.concat(all_werk_matnr_usage_with_forecasts)
        all_werk_matnr_usage_with_forecasts_df.sort_values(by=["WERKS","MATNR","DATE"], inplace=True)
        all_werk_matnr_usage_with_forecasts_df.to_excel("usage_with_forecast.xlsx", index=False)
    else:
        print("[WARNING] No usage found. Thus, no Excel sheet is created!")

    forecast_df.to_excel("raw_forecast.xlsx", index=False)
    
    all_unassigned_forecasts_werk_matnr_df.to_excel("unassigned_forecasts.xlsx", index=False)

# Following cell is ONLY for evaluation

In [101]:
# len(all_unassigned_forecasts_werk_matnr_df)

# Upload to Celonis

In [ ]:
#Create the table in the Data pool

column_config = [
    ColumnTransport(column_name='PBED_MANDT', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='MATNR', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='WERKS', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_PBDNR', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_BDZEI', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_PDATU', column_type=ColumnType.DATE, field_length=26, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_PLNMG', column_type=ColumnType.FLOAT, field_length=15, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_ENTMG', column_type=ColumnType.FLOAT, field_length=15, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_MEINS', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_PLNMGminusPBED_ENTMG', column_type=ColumnType.FLOAT, field_length=15, decimals=None, pk_field=None),
     ColumnTransport(column_name='PBED_PERXX', column_type=ColumnType.STRING, field_length=80, decimals=None, pk_field=None),
     ColumnTransport(column_name='Leftover QTY', column_type=ColumnType.FLOAT, field_length=15, decimals=None, pk_field=None),
     ColumnTransport(column_name='_CELONIS_CHANGE_DATE', column_type=ColumnType.DATE, field_length=23, decimals=None, pk_field=None)
    ]

datapool.create_table(df=all_unassigned_forecasts_werk_matnr_df, table_name="IM_FORECAST_ASSIGNED_QTY", drop_if_exists=True, column_config=column_config)
logger.info('IM_FORECAST_ASSIGNED_QTY has been created and uploaded to the Data Pool')